# Kaggle Digit Recognizer: モデル比較と誤答分析 (Phase 4)

本ノートブックでは、初回提出（ランダムフォレスト: 0.96360）に続き、統計学・機械学習の基本である **「ロジスティック回帰」** を実装し、ランダムフォレストとの公平な比較実験と誤答分析を行います。

### このノートブックで学ぶこと
1. **公平な比較実験の原則:** まったく同じデータ分割（シード値固定）で評価指標（Accuracy）を比較する
2. **ロジスティック回帰（線形モデル）とランダムフォレスト（非線形モデル）の違い:** 統計検定2級の概念との接続
3. **モデルの「視点（重み）」の可視化:** コンピュータが各数字をどのように捉えているかをヒートマップで見る
4. **誤答分析（Error Analysis）:** モデルが見間違えた画像を画面に表示し、「なぜ間違えたのか」を考察する

## 1. ライブラリの読み込みとデータ準備
公平な比較のため、初回と同じ前処理（255.0正規化）と同じデータ分割（`random_state=42`, `stratify=y`）を行います。

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# データパスの自動検索
candidate_paths = [
    "/kaggle/input/competitions/digit-recognizer",
    "/kaggle/input/digit-recognizer",
    "../input/digit-recognizer",
    "./001_digit-recognizer/data",
    "./001_digit-recognizer",
    "./data",
    "."
]

data_dir = None
for p in candidate_paths:
    if os.path.exists(os.path.join(p, "train.csv")):
        data_dir = p
        break

if data_dir is None:
    found = glob.glob("/kaggle/input/**/train.csv", recursive=True)
    if found:
        data_dir = os.path.dirname(found[0])

print(f"データディレクトリ: {data_dir}")
train_df = pd.read_csv(os.path.join(data_dir, "train.csv"))
test_df = pd.read_csv(os.path.join(data_dir, "test.csv"))

# 前処理 (正規化と分割)
y = train_df['label'].values
X = train_df.drop('label', axis=1).values / 255.0
X_test = test_df.values / 255.0

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"準備完了: 訓練データ {X_train.shape}, 検証データ {X_val.shape}")

## 2. モデル1: ロジスティック回帰（Logistic Regression）

> **統計検定2級との接続:**
> ロジスティック回帰は、線形結合 $z = w_1 x_1 + w_2 x_2 + \dots + b$ をシグモイド関数やソフトマックス関数に通して「確率（0〜1）」を出力する統計モデルです。
> 今回は0〜9の10クラス分類なので、**多項ロジスティック回帰（Multinomial Logistic Regression）** を用います。

In [ ]:
print("ロジスティック回帰の学習を開始します...")
# max_iter=200 で十分な収束を確保
lr_model = LogisticRegression(max_iter=200, solver='lbfgs', random_state=42, n_jobs=-1)
lr_model.fit(X_train, y_train)

# 検証データでの予測と正解率
lr_val_preds = lr_model.predict(X_val)
lr_acc = accuracy_score(y_val, lr_val_preds)
print(f"=== ロジスティック回帰 検証正解率: {lr_acc:.4f} ({lr_acc * 100:.2f}%) ===")

### 【ここが面白い！】ロジスティック回帰の「重み（重回帰係数）」を可視化する

ロジスティック回帰の各数字に対する係数（重み $w$）は、まさに **「その数字を判定するために、どの位置のピクセルをプラス評価／マイナス評価しているか」** を表しています。
784個の係数を $28 \times 28$ に戻してヒートマップを描画してみましょう！

In [ ]:
plt.figure(figsize=(15, 6))
# 各クラス (0〜9) の重みを取得
weights = lr_model.coef_

for digit in range(10):
    plt.subplot(2, 5, digit + 1)
    weight_img = weights[digit].reshape(28, 28)
    # 赤: 正の寄与 (ここが黒いとその数字らしい), 青: 負の寄与 (ここが黒いとその数字ではない)
    vmax = max(abs(weight_img.min()), abs(weight_img.max()))
    plt.imshow(weight_img, cmap='seismic', vmin=-vmax, vmax=vmax)
    plt.title(f"Weight for '{digit}'")
    plt.axis('off')

plt.suptitle("Visualizing Learned Weights of Logistic Regression (Red = Positive, Blue = Negative)", fontsize=14)
plt.tight_layout()
plt.show()

**観察ポイント:**
- 「0」の画像: 真ん中が青（中央にインクがあると0ではない）、外枠が赤（周りにインクがあると0らしい）。
- 「1」の画像: 中央の縦ラインだけが赤く、左右が青い。
このように、ロジスティック回帰は「各場所のインクの有無」を単純な足し算・引き算で評価していることが視覚的に理解できます。

## 3. モデル2: ランダムフォレスト（Random Forest）
決定木を多数組み合わせた非線形アンサンブルモデルです。同じデータ分割で学習して比較します。

In [ ]:
print("ランダムフォレストの学習を開始します...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

rf_val_preds = rf_model.predict(X_val)
rf_acc = accuracy_score(y_val, rf_val_preds)
print(f"=== ランダムフォレスト 検証正解率: {rf_acc:.4f} ({rf_acc * 100:.2f}%) ===")

## 4. 2モデルの比較まとめ
同じ検証データ（8,400件）における正解率を比較します。

In [ ]:
comparison_df = pd.DataFrame({
    'Model': ['Logistic Regression (線形モデル)', 'Random Forest (非線形アンサンブル)'],
    'Validation Accuracy': [f"{lr_acc:.4f} ({lr_acc*100:.2f}%)", f"{rf_acc:.4f} ({rf_acc*100:.2f}%)"],
    'Mistakes in Val (8,400問中)': [int((1 - lr_acc) * len(y_val)), int((1 - rf_acc) * len(y_val))]
})
print(comparison_df.to_string(index=False))

# 棒グラフで視覚化
plt.figure(figsize=(6, 4))
bars = plt.bar(['Logistic Regression', 'Random Forest'], [lr_acc, rf_acc], color=['skyblue', 'forestgreen'])
plt.ylim(0.85, 1.0)
plt.ylabel('Validation Accuracy')
plt.title('Accuracy Comparison (Same 8,400 Validation Samples)')
for b in bars:
    yval = b.get_height()
    plt.text(b.get_x() + b.get_width()/2, yval + 0.005, f"{yval:.4f}", ha='center', fontweight='bold')
plt.show()

### なぜランダムフォレストの方が約4%（約360問分）も正解が多いのか？
- **ロジスティック回帰（線形）:** 各ピクセルの単独の濃淡しか見られません。「ピクセルAとピクセルBの両方が黒い時だけ角（コーナー）になる」といった複雑な条件を表現できません。
- **ランダムフォレスト（非線形）:** 決定木の分岐によって「ピクセルAが黒 かつ ピクセルBが白」のような複数ピクセルの組み合わせパターンを学習できるため、手書き文字の複雑な曲がりや線の繋がりを正しく認識できます。

## 5. 誤答分析（Error Analysis）: モデルはなぜ間違えたのか？

統計検定2級でも重要な **「混同行列（Confusion Matrix）」** を用いて、ランダムフォレストが間違えたパターンの内訳を調査します。

In [ ]:
cm = confusion_matrix(y_val, rf_val_preds)

# 正解の対角線を 0 にして、「間違えた部分だけ」を目立たせる
cm_errors_only = cm.copy()
np.fill_diagonal(cm_errors_only, 0)

plt.figure(figsize=(9, 7))
sns.heatmap(cm_errors_only, annot=True, fmt='d', cmap='Reds')
plt.title("Confusion Matrix: Mistakes Only (Random Forest)")
plt.xlabel("Predicted Label (モデルの予想)")
plt.ylabel("True Label (本当の正解)")
plt.show()

### 実際にモデルが間違えた画像を画面に表示してみる
「人間なら読めるのか？」「それとも悪筆すぎて人間でも迷うのか？」を目で見て確かめてみましょう。

In [ ]:
# 間違えたサンプルのインデックスを抽出
mistake_indices = np.where(y_val != rf_val_preds)[0]
print(f"検証データ 8,400枚中、間違えた画像は合計 {len(mistake_indices)} 枚です。")

# 最初の10枚をプロット
plt.figure(figsize=(12, 5))
for i, idx in enumerate(mistake_indices[:10]):
    plt.subplot(2, 5, i + 1)
    img = X_val[idx].reshape(28, 28)
    true_label = y_val[idx]
    pred_label = rf_val_preds[idx]
    
    plt.imshow(img, cmap='gray')
    plt.title(f"True: {true_label} | Pred: {pred_label}", color='red')
    plt.axis('off')

plt.suptitle("Examples of Misclassified Images (True vs Predicted)", fontsize=14)
plt.tight_layout()
plt.show()

### 考察まとめ
1. **人間でも紛らわしい文字:**
   - 線が途切れて「4」に見える「9」や、「3」と「8」、「5」と「3」など、書いた本人の癖が強く人間でも判別に迷う画像が多く含まれています。
2. **今後の改善の方向性（CNNへの期待）:**
   - 人間は文字を「ピクセルの絶対位置」ではなく「ループ（輪っか）があるか」「尖った角があるか」といった局所的な特徴（特徴マップ）で見ています。
   - このような「局所的な形状のパターン」を捉えるのが、まさに画像認識の主役である **畳み込みニューラルネットワーク（CNN）** です。